In [89]:
%pip install selenium-stealth

Note: you may need to restart the kernel to use updated packages.


In [2]:
import urllib.request
import requests
import bs4
from bs4 import BeautifulSoup
import time
import random
import undetected_chromedriver as uc
import pandas as pd
import numpy as np

In [2]:
driver = uc.Chrome()

driver.get("https://gizmodo.com/latest/page/2750")

time.sleep(5)
html = driver.page_source

In [28]:
html

'<html lang="en-US" class="dark"><head><script type="text/javascript" async="" src="https://cdn.id5-sync.com/api/1.0/id5PrebidModule.js"></script>\n    <meta charset="utf-8">\n    <meta name="viewport" content="width=device-width, initial-scale=1">\n    <link rel="icon" type="image/png" href="/favicon.png?v1" sizes="96x96">\n    <link rel="icon" type="image/svg+xml" href="/favicon.svg?v1">\n    <link rel="shortcut icon" href="/favicon.ico?v1">\n    <link rel="apple-touch-icon" sizes="180x180" href="/apple-touch-icon.png?v1">\n    <link rel="manifest" href="/site.webmanifest">\n    <link rel="mask-icon" href="/safari-pinned-tab.svg?v1" color="#004fff">\n    <meta name="apple-mobile-web-app-title" content="Gizmodo">\n    <meta name="application-name" content="Gizmodo">\n    <meta name="msapplication-TileColor" content="#004fff">\n    <meta name="theme-color" content="#ffffff">\n    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/5.15.3/css/all.min.css">\n

In [11]:
links = []

In [12]:
for i in range(1, 2751):
    url = f"https://gizmodo.com/latest/page/{i}"
    print(f"Scraping: {url} links:{len(links)}")

    try:
        driver.get(url)
        time.sleep(random.uniform(2.5, 6.0)) 
        html = driver.page_source

        if "Just a moment" in html:
            print("Cloudflare challenge detected. Waiting...")
            time.sleep(10)
            html = driver.page_source
            if "Just a moment" in html:
                print("Still blocked. Skipping page.")
                continue
        
        soup = BeautifulSoup(html, "html.parser")

        articles = soup.select("div.mb-5.pb-5.border-b-0\\.5")  # each article container
        
        for article in articles:
            urls = article.select("div.h-full.w-full.md\\:w-2\\/3.flex.flex-col.gap-4.md\\:gap-0.justify-between a.block")
            for url in urls:
                links.append(url.get("href"))

        
    except Exception as e:
        print("Error:", e)
        time.sleep(10)
        continue

Scraping: https://gizmodo.com/latest/page/1 links:0
Scraping: https://gizmodo.com/latest/page/2 links:20
Scraping: https://gizmodo.com/latest/page/3 links:40
Scraping: https://gizmodo.com/latest/page/4 links:60
Scraping: https://gizmodo.com/latest/page/5 links:80
Scraping: https://gizmodo.com/latest/page/6 links:100
Scraping: https://gizmodo.com/latest/page/7 links:120
Scraping: https://gizmodo.com/latest/page/8 links:140
Scraping: https://gizmodo.com/latest/page/9 links:160
Scraping: https://gizmodo.com/latest/page/10 links:180
Scraping: https://gizmodo.com/latest/page/11 links:200
Scraping: https://gizmodo.com/latest/page/12 links:220
Scraping: https://gizmodo.com/latest/page/13 links:240
Scraping: https://gizmodo.com/latest/page/14 links:260
Scraping: https://gizmodo.com/latest/page/15 links:280
Scraping: https://gizmodo.com/latest/page/16 links:300
Scraping: https://gizmodo.com/latest/page/17 links:320
Scraping: https://gizmodo.com/latest/page/18 links:340
Scraping: https://gizmodo

In [59]:
len(titles)

12

In [20]:
df = pd.DataFrame(data={'titles':titles,'links':links,'dates':dates})

In [21]:
df.to_csv('ahrefs_1.csv',index=False)

In [73]:
len(links)

20

In [61]:
len(dates)

12

In [62]:
dates

[None, None, None, None, None, None, None, None, None, None, None, None]

In [63]:
titles

['‘Alexa, Pay for Gas’ Is Amazon’s Latest IoT Honey Trap',
 '‘Alexa, Pay for Gas’ Is Amazon’s Latest IoT Honey Trap',
 'This Beautiful Smartphone Is Like an Affordable Samsung Galaxy S10',
 'The First Folding PC Is Remarkably Smart',
 'A Bed That Cools and Heats Each Sleeper Separately Will Save Countless Relationships',
 'A Smart Wooden Stick Is Now a Thing That Exists',
 'Samsung Might Have Just Made the Nicest Chromebook Ever',
 'Why Did the Pentagon Push the FDA to Approve a Powerful New Opioid?',
 'Disney+ Edits Out Goofy’s Smoking Scene in Supposedly Uncut Version of Classic Movie',
 'During Brain Surgery, This AI Can Diagnose a Tumor in 2 Minutes',
 'This Company Just Gave Dudes a Good Reason to Shave Their Taints',
 'What Was the Most Fun Thing Humans Could Do 5,000 Years Ago?']

In [13]:
len(links)

54961

In [14]:
df = pd.DataFrame(data={"links":links},columns=["links"])

In [15]:
df.head()

,links
0,https://gizmodo.com/10-things-to-remember-abou...
1,https://gizmodo.com/bitcoin-price-plunges-six-...
2,https://gizmodo.com/this-stunning-predator-bad...
3,https://gizmodo.com/doctor-who-war-between-lan...
4,https://gizmodo.com/california-startup-flies-m...


In [16]:
df.to_csv('gizmodo_links.csv',index=False)

In [18]:
df['links'].iloc[0]

'https://gizmodo.com/10-things-to-remember-about-five-nights-at-freddys-2000685629'

In [1]:
import json
import multiprocessing
import random
from datetime import datetime

In [3]:
df = pd.read_csv('gizmodo_links.csv')

In [4]:
links=df['links']
len(links)
links = df['links'].tolist()

In [6]:
idxs = [f"{s[0]}-{s[-1]}" for s in np.array_split(np.arange(len(links)), 24)]

In [65]:
quarters = [(1,3),(4,6),(7,9),(10,12)]

In [32]:
from selenium_stealth import stealth
from selenium.webdriver.remote.remote_connection import RemoteConnection
def new_driver():
    options = uc.ChromeOptions()
    driver = uc.Chrome(
        use_subprocess=True,
        options=options,
        client_config={
            "connection_timeout": 300,  
            "read_timeout": 300          
        }
    )
    return driver
driver=new_driver()

In [5]:
df_final = pd.DataFrame(columns=["titles", "contents", "links","dates"])
df_any = pd.DataFrame(columns=["titles", "contents", "links","dates","year","month","quarter"])

In [23]:
df_final = pd.read_csv('gizmodo_contents.csv')

In [66]:
def init_year_hashmap(year):
    return {q: pd.DataFrame(columns=["titles", "contents", "links", "dates", "year", "month", "quarter"]) for q in range(1, 5)}

df_any_map = {} 

for y in range(2020,2025):
    df_any_map[y] = init_year_hashmap(y)



In [76]:
def scrape_links(links):
    global driver, df_final, df_any_map
    saved_quarters = set([(2024, 3),(2024, 2)])
    chosen_links = []
    display_handle = display("", display_id=True)
    def year_full(year):
        return all(len(df_any_map.get(year, {}).get(q, pd.DataFrame())) >= 500 for q in range(1, 5))

    i = 0
    while i < len(links):
        url = links[i]

        if year_full(2020):
            print("Year 2020 fully scraped. Stopping.")
            break
    
        try:
            if url in chosen_links:
                i += 1
                continue
            chosen_links.append(url)
    
            driver.get(url)
            time.sleep(random.uniform(3.5, 7.0))
            html = driver.page_source
    
            if "Just a moment" in html:
                driver.quit()
                driver = new_driver()
                driver.get(url)
                time.sleep(random.uniform(3.5, 7.0))
                html = driver.page_source
                if "Just a moment" in html:
                    driver.quit()
                    driver = new_driver()
                    print("Blocked, skipping page")
                    i += 1
                    continue
    
            soup = BeautifulSoup(html, "html.parser")
            datetag = soup.select_one(
                'time.updated.typo-space-h5.md\\:typo-space-h3.uppercase.text-grey-600.dark\\:text-grey-200'
            )
            if not datetag:
                i += 1
                continue
            date = datetime.fromisoformat(datetag['datetime'])
            month = date.month
            year = date.year
            quarter = (month - 1)//3 + 1
    
            # Build entry
            title = soup.select_one('h1.typo-sofia-h4.md\\:typo-sofia-h3')
            subtitle = soup.select_one('div.post-excerpt.mt-5.typo-space-body-2.md\\:typo-space-body-1')
            content = soup.select_one('article.post')
            content_text = ""
            if subtitle: content_text += subtitle.get_text(" ", strip=True) + " "
            if content: content_text += content.get_text(" ", strip=True)
    
            entry = {
                "titles": title.get_text(" ", strip=True) if title else "",
                "contents": content_text,
                "links": url,
                "dates": date,
                "year": year,
                "month": month,
                "quarter": quarter
            }
    
            if year not in df_any_map:
                df_any_map[year] = {q: pd.DataFrame(columns=df_final.columns) for q in range(1,5)}
            local_count_year = sum(len(df_any_map[year][q]) for q in range(1,5))
            display_handle.update(
                f"Scraped: {url} | Year: {year} Q{quarter} | Local Year:{local_count_year}  Q: {len(df_any_map[year][quarter])}"
            )
            if url in df_any_map[year][quarter]['links'].values:
                i+=1
                continue
            if (year, quarter) not in saved_quarters:
                df_any_map[year][quarter] = pd.concat(
                    [df_any_map[year][quarter], pd.DataFrame([entry])], ignore_index=True
                )
                if len(df_any_map[year][quarter]) >= 500:
                    df_final = pd.concat([df_final, df_any_map[year][quarter]], ignore_index=True)
                    df_final.to_csv('gizmodo_contents.csv', index=False)
                    print(f"Quarter reached 500 entries: Year {year} Q{quarter}, saved to CSV")
                    saved_quarters.add((year, quarter))
                    i += 100
                else:
                    i += 1
                    
            else:
                i += 100
                continue
    
        except Exception as e:
            print("Error:", e)
            if "Read timed out" in str(e):
                driver.quit()
                driver = new_driver()
            time.sleep(5)
            i += 1

In [77]:
scrape_links(links[13000:])

'Scraped: https://gizmodo.com/trump-approves-bill-barring-rural-telecom-carriers-from-1842302739 | Year: 2020 Q1 | Local Year:1999  Q: 499'

Error: HTTPConnectionPool(host='localhost', port=59179): Read timed out. (read timeout=120)
Quarter reached 500 entries: Year 2024 Q1, saved to CSV
Blocked, skipping page
Blocked, skipping page
Blocked, skipping page
Blocked, skipping page
Quarter reached 500 entries: Year 2023 Q4, saved to CSV


E:\Temp\ipykernel_28312\4226795174.py:80: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_any_map[year][quarter] = pd.concat(


Quarter reached 500 entries: Year 2023 Q3, saved to CSV


E:\Temp\ipykernel_28312\4226795174.py:80: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_any_map[year][quarter] = pd.concat(


...

Quarter reached 500 entries: Year 2020 Q3, saved to CSV


E:\Temp\ipykernel_28312\4226795174.py:80: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_any_map[year][quarter] = pd.concat(


Error: HTTPConnectionPool(host='localhost', port=51877): Read timed out. (read timeout=120)
Quarter reached 500 entries: Year 2020 Q2, saved to CSV


E:\Temp\ipykernel_28312\4226795174.py:80: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_any_map[year][quarter] = pd.concat(


Quarter reached 500 entries: Year 2020 Q1, saved to CSV
Year 2020 fully scraped. Stopping.


In [81]:
for i in [2024,2023,2022,2021,2020]:
    for j in [1,2,3,4]:
        print(df_any_map[i][j].shape)

(501, 7)
(501, 7)
(0, 7)
(0, 7)
(500, 7)
(500, 7)
(500, 7)
(500, 7)
(500, 7)
(500, 7)
(500, 7)
(500, 7)
(500, 7)
(500, 7)
(500, 7)
(500, 7)
(500, 7)
(500, 7)
(500, 7)
(500, 7)


In [26]:
df_final.iloc[:-1000]

,titles,contents,links,dates,year,month,quarter
0,"<h1 class=""typo-sofia-h4 md:typo-sofia-h3"">\n ...","This Thanksgiving season, feed your 'Frankenst...",https://gizmodo.com/you-can-read-guillermo-del...,2025-10-25 16:15:23-04:00,NaN,NaN,NaN
1,"<h1 class=""typo-sofia-h4 md:typo-sofia-h3"">\n ...","'Five Nights at Freddy's 2', starring Josh Hut...",https://gizmodo.com/five-nights-at-freddys-2-t...,2025-11-05 09:00:48-05:00,NaN,NaN,NaN
2,"<h1 class=""typo-sofia-h4 md:typo-sofia-h3"">\n ...",The company says it’s adding new “guardrails” ...,https://gizmodo.com/openai-blocks-users-from-m...,2025-10-17 10:40:25-04:00,NaN,NaN,NaN
3,"<h1 class=""typo-sofia-h4 md:typo-sofia-h3"">\n ...",Adam Raine's family members added troubling ne...,https://gizmodo.com/openai-weakened-chatgpts-s...,2025-10-23 07:05:31-04:00,NaN,NaN,NaN
4,"<h1 class=""typo-sofia-h4 md:typo-sofia-h3"">\n ...",It's against Facebook's terms of service to re...,https://gizmodo.com/facebook-removes-ice-track...,2025-10-14 16:15:01-04:00,NaN,NaN,NaN
...,...,...,...,...,...,...,...
2495,"<h1 class=""typo-sofia-h4 md:typo-sofia-h3"">\n ...",The iPhone 16 didn't disappoint and IFA 2024 g...,https://gizmodo.com/the-best-gadgets-of-septem...,2024-10-01 09:45:43-04:00,NaN,NaN,NaN
2496,"<h1 class=""typo-sofia-h4 md:typo-sofia-h3"">\n ...","It's coming to the Sony WH-1000XM5, WF-1000XM5...",https://gizmodo.com/sonys-headphones-are-getti...,2024-10-04 17:15:35-04:00,NaN,NaN,NaN
2497,"<h1 class=""typo-sofia-h4 md:typo-sofia-h3"">\n ...",NASA selected two space observatory proposals ...,https://gizmodo.com/one-of-these-two-space-obs...,2024-10-08 13:45:27-04:00,NaN,NaN,NaN
2498,"<h1 class=""typo-sofia-h4 md:typo-sofia-h3"">\n ...",The yet-to-be-named comet will make its closes...,https://gizmodo.com/new-sungrazing-comet-may-o...,2024-10-01 15:15:46-04:00,NaN,NaN,NaN


In [131]:
df_any['year']=df_any['dates'].map(lambda x:x.year)
df_any['month']=df_any['dates'].map(lambda x:x.month)

In [133]:
df_any['quarter'] = ((df_any['month'] - 1) // 3 + 1)

In [141]:
df_any.head()

,titles,contents,links,dates


In [139]:
df_any.to_csv('gizmodo_any.csv', index=False)

In [168]:
df_any

,titles,contents,links,dates,year,month,quarter
0,"[\n Cops Pose Before a $200,000 Mount...",A three-month investigation into retail theft ...,https://gizmodo.com/cops-pose-before-a-200000-...,2024-07-10 09:50:23-04:00,NaN,NaN,NaN
1,"[\n No, That Acolyte Planet Is Not Wh...",Leslye Headland has seen your speculation abou...,https://gizmodo.com/star-wars-the-acolyte-unkn...,2024-07-05 18:30:10-04:00,NaN,NaN,NaN
2,"[\n, [Fallout’s], Vinyl Soundtrack Is Coming,...",Ella Purnell and Walton Goggins star in the Pr...,https://gizmodo.com/fallout-tv-show-vinyl-musi...,2024-04-24 13:00:01-04:00,NaN,NaN,NaN
3,[\n Mark Hamill Talks About His Hairy...,The Star Wars legend was part of a San Diego C...,https://gizmodo.com/sdcc-2024-mark-hamill-drea...,2024-07-28 17:15:35-04:00,NaN,NaN,NaN
4,[\n The Best Gadgets of June 2024\n ...,There's no time for summertime sadness with ga...,https://gizmodo.com/best-gadgets-of-june-2024-...,2024-07-01 22:05:01-04:00,NaN,NaN,NaN
...,...,...,...,...,...,...,...
1972,[\n Chromecast With Google TV Might F...,A new dongle for the 4K streaming crowd could ...,https://gizmodo.com/google-tv-chromecast-4k-18...,2024-04-23 15:10:01-04:00,NaN,NaN,NaN
1973,[\n Amazon Is Ready For Even More of ...,"Plus, Doctor Who 's Russell T Davies teases w...",https://gizmodo.com/the-boys-season-5-amazon-p...,2024-05-14 10:30:00-04:00,NaN,NaN,NaN
1974,"[\n Lego’s New , [Jaws], Set Will No...","The 1,497-piece tribute to the Steven Spielber...",https://gizmodo.com/lego-ideas-jaws-set-price-...,2024-07-04 09:00:33-04:00,NaN,NaN,NaN
1975,"[\n First Trailer for , [Rick and Mor...",Adult Swim's riff on the blockbuster animated ...,https://gizmodo.com/first-trailer-for-rick-and...,2024-07-11 14:45:32-04:00,NaN,NaN,NaN


In [82]:
df_test = pd.read_csv('gizmodo_contents.csv')

In [83]:
df_test

,titles,contents,links,dates,year,month,quarter
0,"<h1 class=""typo-sofia-h4 md:typo-sofia-h3"">\n ...","This Thanksgiving season, feed your 'Frankenst...",https://gizmodo.com/you-can-read-guillermo-del...,2025-10-25 16:15:23-04:00,NaN,NaN,NaN
1,"<h1 class=""typo-sofia-h4 md:typo-sofia-h3"">\n ...","'Five Nights at Freddy's 2', starring Josh Hut...",https://gizmodo.com/five-nights-at-freddys-2-t...,2025-11-05 09:00:48-05:00,NaN,NaN,NaN
2,"<h1 class=""typo-sofia-h4 md:typo-sofia-h3"">\n ...",The company says it’s adding new “guardrails” ...,https://gizmodo.com/openai-blocks-users-from-m...,2025-10-17 10:40:25-04:00,NaN,NaN,NaN
3,"<h1 class=""typo-sofia-h4 md:typo-sofia-h3"">\n ...",Adam Raine's family members added troubling ne...,https://gizmodo.com/openai-weakened-chatgpts-s...,2025-10-23 07:05:31-04:00,NaN,NaN,NaN
4,"<h1 class=""typo-sofia-h4 md:typo-sofia-h3"">\n ...",It's against Facebook's terms of service to re...,https://gizmodo.com/facebook-removes-ice-track...,2025-10-14 16:15:01-04:00,NaN,NaN,NaN
...,...,...,...,...,...,...,...
13998,Bill Barr Recently Met With Australian Officia...,"Peter Dutton, Australia’s Home Affairs Ministe...",https://gizmodo.com/bill-barr-recently-met-wit...,2020-03-13 05:30:00-04:00,2020.0,3.0,1.0
13999,An Explosive Clone Wars Episode Asks the Price...,There’s a moment in this week’s Star Wars: The...,https://gizmodo.com/an-explosive-clone-wars-ep...,2020-03-13 03:01:00-04:00,2020.0,3.0,1.0
14000,Sennheiser’s New Wireless Earbuds Up the Ante ...,Sennheiser’s Momentum True Wireless earbuds de...,https://gizmodo.com/sennheisers-new-wireless-e...,2020-03-13 03:00:01-04:00,2020.0,3.0,1.0
14001,"Google, Facebook Contractors Still Expected On...",The tumultuous divide between full-time staff ...,https://gizmodo.com/google-facebook-contractor...,2020-03-13 02:36:00-04:00,2020.0,3.0,1.0
